In [ ]:
# Where are critical events occuring?

# 4knews + GNEWS -------------------------------------------------------------
# https://github.com/AndyTheFactory/newspaper4k
# https://newspaper4k.readthedocs.io/en/latest/
# Use an aggregator such as https://gnews.io/

# Use Anthropic to extract sites from the news texts

#-----------------------------------------------------------------------
# RTS, May 23, 2025
#-------------------------------------------------------------------------------

# DONE
# get urls from news sites reporting on climate crisis
# get text describing instances of crisis
# get sites directly impacted by the events/crisis

# TODO
# filter sites
# get GPS coordinates from those locations via reverse geolocation lookup (for satellite imagery collection)
# add to collection (test as CSV, later DB): location_name, latitude/longitude coordinates, news_source, date

In [1]:
# variables specific to your CoLab setup
from google.colab import drive
import os, sys
drive.mount('/content/drive')
root = '/content/drive/MyDrive/'
sys.path.append(root +"Colab/research/code/")
datapath = root + "Colab/research/data/"
datapathnudge = root + "Projects/Nudge/geo_images/samples/"
datapathcities = root + "Projects/Nudge/sites/"

Mounted at /content/drive


In [2]:
# install packages
%%capture
!pip install newspaper4k
!pip install --upgrade lxml_html_clean
!pip install reverse_geocode --upgrade
!pip install -qU langchain-anthropic

In [3]:
# imports
import os, requests
import newspaper
from newspaper import Article, Source, fulltext
import nltk

In [4]:
# API keys
from google.colab import userdata
os.environ["GNEWS_API_KEY"] = userdata.get('GNews')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic_key')

In [5]:
def fetch_articles(term, max_items = 12):
    params = {
        'q': term,
        'token': os.environ["GNEWS_API_KEY"],
        'lang': 'en',
        'max': max_items
    }
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    results =  [article['url'] for article in data.get('articles', [])]
    return(results)

In [6]:
#SEARCH_TERMS = ["environment", "pollution", "climate", "crisis", "heat", "flood", "draught"]
#SEARCH_TERMS = ["pollution", "crisis", "heat", "flood", "draught"]
SEARCH_TERMS = ["climate crisis", "open pit mining"]
BASE_URL = "https://gnews.io/api/v4/search"

results = []

urls = set()
for term in SEARCH_TERMS:
    print(f"Searching for: {term}")
    urls.update(fetch_articles(term))

    for url in urls:
        results.append([term, url])

print("Total # of detected instances: ", len(results))

Searching for: climate crisis
Searching for: open pit mining
Number of detected instances:  30


In [7]:
# Get all the results and find locations mentioned in them.

limit = 2048
basket = []
harvest = []

for r in results:
    try:
        url = r[1]
        article = Article(url)
        article.download()
        article.parse()

        response = article.text[:limit]
        results = response.split('.')
        harvest.append(results)

        for item in results[:-1]:
            item = item.strip() + '.'
            basket.append(item)
    except:
        print("Error occured at: ", url)

    harvest.append(basket)

print(len(harvest))

60


In [8]:
# Interface to Anthropic Claude
from langchain_anthropic import ChatAnthropic
model_type = "claude-3-5-sonnet-20240620"
model = ChatAnthropic(model=model_type)

In [34]:
# Improve the query to get more precise results that satellite imagery can respond to...
# limit the number of requests to Antropic

selection = harvest[:2]
for fruit in selection:
    text = "".join(fruit)

    # try these keys out
    key = "open pit mining"
    #key = "clear cut burning"
    #key = "deforestation"
    #key = "fires"
    #key = "urban sprawl"
    #key = "urban slums"
    #key = "monocultures" #"green deserts"

    #query = "List only sites refered to as impacted by climate crisis in this text: " + text + "\n list the nearest town or city to the site.  Omit all other information."
    #query = "List only sites refered to as impacted by " + key +  " in this text: " + text + "\n list the nearest town or city to the site.  Omit all other information."
    #query = "List only locations showing signs of " + key +  " in this text: " + text + "\n list the nearest town or city to the site.  Omit all other information."

    #query = "List only locations showing signs of " + key +  " in this text: " + text + "\n list the exact name of the site with province, country, latitude and longitude. DO NOT list anything else."
    query = "List only locations showing signs of " + key +  " in this text: " + text + "\n list the exact name of the site with province, country. DO NOT list anything else."

    result = model.invoke(query)
    print("----------------------------------------------------------------------"*2)
    print(fruit)
    print("-> " + result.content)
    print()

--------------------------------------------------------------------------------------------------------------------------------------------
['Hawaii’s governor signed legislation that boosts a tax imposed on hotel room and vacation rental stays in order to raise money to address the consequences of the climate crisis', '\n\nIt’s the first time in a government in the US imposes such levy to help cope with a warming planet', '\n\nOfficials estimate the tax will generate nearly $100m annually', ' The money will be used for projects such as replenishing sand on eroding Waikiki beaches, promoting the use of hurricane clips to secure roofs during powerful storms and clearing flammable invasive grasses like those that fueled the large wildfire that killed 102 people on the island of Maui two years ago', '\n\nHawaii’s governor, Josh Green, said on Tuesday that other states and nations will need to act similarly to address climate disasters roiling the planet', '\n\n“There will be no way to de

In [38]:
if(result.content != ""):
    detected = (result.content.split(':')[1])

print(detected)



1. Grassy Mountain, Alberta, Canada
2. Great Bear, Red Lake, Ontario, Canada


In [40]:
#collected_query = "Find the location of the " + key  +  " in " + detected + " List longitude and latidude of that location if possible. DO NOT include other information."
collected_query = "Find the location of the " + key  +  " in Grassy Mountain, Alberta, Canada. List longitude and latidude of that location if possible. DO NOT include other information."

result = model.invoke(collected_query)
print("-> " + result.content)

-> The open pit mining project at Grassy Mountain, Alberta, Canada is located at approximately:

Latitude: 49.5778° N
Longitude: 114.4872° W


In [ ]:
# If cities or locations are mentioned in the texts, extract them and perform reverse geocoding -> site variable
# example

import reverse_geocode

site = [56.55813813197379, 63.984078654435166]
latitude = site[0]
longitude = site[1]
coordinates = latitude, longitude
site = reverse_geocode.get(coordinates)
print(site)
print("Closest city: ", site['city'])

{'country_code': 'RU', 'city': 'Butka', 'latitude': 56.71788, 'longitude': 63.78661, 'population': 3510, 'state': 'Sverdlovsk Oblast', 'country': 'Russian Federation'}
Closest city:  Butka
